# Procesamiento del Lenguaje Natural II — Tema 1

**Objetivos (5 ejercicios):**
1) **IR — Mini-buscador con BM25:** implementar la función BM25 y un ranker top-k sobre un mini-corpus.
2) **NER — Métrica con solape (IoU):** calcular P/R/F1 con emparejamiento *greedy* por traslape de spans.
3) **WSD — Lesk simplificado:** desambiguar sentidos por solape contexto↔glosas normalizadas.
4) **Correferencia de pronombres (heurística):** resolver pronombres por recencia + concordancia y evaluar por pares.
5) **QA factoidal mini (SVO):** extraer Sujeto-Verbo-Objeto y responder “¿Quién VERBO OBJETO?”.


# NLP II — Tema 1: Fundamentos y recursos (1.1–1.5)

Este cuaderno cubre, con práctica en Python, los apartados:
1. **Introducción y repaso de PLN I**
2. **Análisis morfológico, sintáctico y semántico**
3. **Representaciones clásicas (BoW, TF‑IDF, WordNet)**
4. **Recuperación de información (IR)**
5. **Extracción de información (IE)**

Incluye **implementaciones básicas**, **ejercicios autocorregibles** y **soluciones** (ocultables).  
**Sugerencia:** recorre en orden y ejecuta cada celda.


In [30]:
# Cambia a True para cargar automaticamente las soluciones de cada ejercicio
USE_SOLUTIONS = False


## 0) Setup y utilidades

In [31]:

import math, re, random, unicodedata, itertools
from collections import Counter, defaultdict
from dataclasses import dataclass
random.seed(42)

def strip_accents(s: str) -> str:
    """Quita tildes/acentos preservando letras."""
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

def normalize_whitespace(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

def sent_tokenize(text: str):
    # Segmentacion muy simple por punto/exclamacion/interrogacion
    sents = re.split(r"(?<=[\.!?])\s+", text.strip())
    return [s for s in sents if s]

def is_emoji(ch: str) -> bool:
    return ch in ("😀😁😂🤣😃😄😅😆😉😊🙂🙃😍😘😗😙😚😋😛😜😝🤪🤨🧐🤓😎"
                  "🤩🥳😏😒🙄😬🤥😌😔😪🤤😴😷🤒🤕🤧🥵🥶🥴😵🤯🤠")  # demo


## 1) Mini-corpus en espanol (offline)

In [32]:

CORPUS = [
    {
        "id": 1,
        "title": "Economia: inflacion y empleo",
        "text": "La economia espanola registra un descenso de la inflacion y una ligera mejora del empleo en julio. "
                "El Banco de Espana advierte de la incertidumbre internacional."
    },
    {
        "id": 2,
        "title": "Deportes: liga de futbol",
        "text": "El Real Madrid vencio 2-0 al Valencia en el Santiago Bernabeu. Vinicius marco el primer gol del partido."
    },
    {
        "id": 3,
        "title": "Tecnologia: ciberseguridad",
        "text": "Un informe revela vulnerabilidades en dispositivos IoT domesticos. Expertos recomiendan actualizar el firmware con frecuencia."
    },
    {
        "id": 4,
        "title": "Ciencia: energia solar",
        "text": "Un nuevo material fotovoltaico promete mejorar la eficiencia de los paneles solares en climas nublados."
    },
    {
        "id": 5,
        "title": "Salud: habitos y bienestar",
        "text": "Dormir entre 7 y 9 horas reduce el riesgo de enfermedades cardiovasculares, segun un estudio multicentrico."
    },
    {
        "id": 6,
        "title": "Politica: presupuesto",
        "text": "El Congreso aprueba el presupuesto con apoyo de varios grupos, aunque la oposicion critica el incremento del gasto."
    },
    {
        "id": 7,
        "title": "Cultura: festival de cine",
        "text": "El Festival de San Sebastian inaugura su edicion con una pelicula espanola a competicion."
    },
    {
        "id": 8,
        "title": "Clima: alerta por lluvias",
        "text": "La AEMET declara alerta naranja por lluvias intensas en varias provincias del norte peninsular."
    },
]
len(CORPUS)


8


## 1.1) Preprocesado y tokenizacion (ES)

Objetivo: obtener tokens robustos para espanol, tratando:
- Acentos (informatica vs informática),
- Contracciones (del, al),
- Cliticos (p. ej., damelo, explicaselo),
- Signos, URLs, @handles, #hashtags, numeros y separadores locales.


In [33]:

URL_RE = r"(?:https?://\S+|www\.\S+)"  # no-capturante
HANDLE_RE = r"@[A-Za-z0-9_]+"
HASHTAG_RE = r"#[\wáéíóúÁÉÍÓÚñÑ]+"
NUM_RE = r"\d+(?:[\.,]\d+)*"
WORD_RE = r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]+(?:-[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]+)*"

TOKEN_RE = re.compile(
    "|".join([URL_RE, HANDLE_RE, HASHTAG_RE, NUM_RE, WORD_RE, r"[^\s]"]),
    re.UNICODE
)

# Heuristico simple de segmentacion de cliticos comunes en ES
CLITICS = [
    "me", "te", "se", "lo", "la", "le", "los", "las", "les", "nos", "os"
]

def segment_clitics(token: str):
    """
    Segmenta encliticos frecuentes en ES (heuristico).
    Ejemplos: 'damelo' -> ['dame','lo'] o ['da','me','lo'].
    Estrategia: buscar verbos con enclitico o terminaciones de imperativo/infinitivo + clitico(s).
    """
    t = token
    low = t.lower()
    # 1) separa "-se/-me/-lo..." si hay guion
    if "-" in t and any(low.endswith("-"+c) for c in CLITICS):
        parts = t.split("-")
        if len(parts) >= 2 and parts[-1].lower() in CLITICS:
            return ["-".join(parts[:-1]), parts[-1]]
    # 2) separa acumulacion de 1-2 cliticos al final (max 2 por simplicidad)
    for c2 in CLITICS:
        if low.endswith(c2):
            stem = t[:len(t)-len(c2)]
            low_stem = stem.lower()
            for c1 in CLITICS:
                if low_stem.endswith(c1):
                    base = stem[:len(stem)-len(c1)]
                    if len(base) >= 2:
                        return [base, stem[len(base):], t[len(stem):]]
            if len(stem) >= 2:
                return [stem, t[len(stem):]]
    return [t]

def tokenize_es(text: str, lowercase=False, strip_acc=False, segment_enclitics=True):
    text = normalize_whitespace(text)
    tokens = TOKEN_RE.findall(text)
    out = []
    for tok in tokens:
        segs = [tok]
        if segment_enclitics and re.search(r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]", tok):
            segs = segment_clitics(tok)
        for s in segs:
            if lowercase:
                s = s.lower()
            if strip_acc:
                s = strip_accents(s)
            out.append(s)
    return out

# Pruebas rapidas
samples = [
    "Damelo ahora, por favor.",
    "Explicaselo a @juan en https://urjc.es",
    "Compre 1.234,56 EUR en 3-4 cuotas.",
    "Vamonos a Madrid!"
]
for s in samples:
    print(s, "->", tokenize_es(s, lowercase=True, strip_acc=False, segment_enclitics=True))


Damelo ahora, por favor. -> ['da', 'me', 'lo', 'ahora', ',', 'por', 'favor', '.']
Explicaselo a @juan en https://urjc.es -> ['explica', 'se', 'lo', 'a', '@juan', 'en', 'https://urjc.es']
Compre 1.234,56 EUR en 3-4 cuotas. -> ['compre', '1.234,56', 'eur', 'en', '3', '-', '4', 'cuotas', '.']
Vamonos a Madrid! -> ['vamo', 'nos', 'a', 'madrid', '!']



## 1.2) Lematizacion y etiquetado morfosintactico (baseline)

Implementamos una lematizacion sencilla por reglas (sufijos comunes) y un POS-tagging heuristico
para ilustrar el flujo. Nota: es un demo educativo (no pretende competir con spaCy/Stanza).


In [34]:

# Reglas de lema (muy simplificadas)
VERB_SUFFIXES = ["arme","arte","arse","irme","irte","irse","arnos","aros","aron","are","aras","ara","aremos","areis","aran",
                 "erme","erte","erse","iremos","ereis","eran",
                 "irme","irte","irse","amos","aron","iendo","ando","are","ere","ire",
                 "ar","er","ir"]
NOUN_ADJ_SUFFIXES = ["es", "s", "mente", "ico", "ica", "osa", "oso", "itos", "itas", "ito", "ita", "cion", "ciones"]

def lemmatize_simple(token: str) -> str:
    low = token.lower()
    # Verbos infinitivo aproximado
    for suf in ["ar","er","ir"]:
        if low.endswith(suf) and len(low) > len(suf)+1:
            return low
    # Gerundios y participios comunes
    for suf in ["ando","iendo","ado","ido"]:
        if low.endswith(suf) and len(low) > len(suf)+1:
            return low[:-len(suf)] + ("ar" if suf.endswith("ando") else "er")
    # Plurales y adverbios simples
    for suf in NOUN_ADJ_SUFFIXES:
        if low.endswith(suf) and len(low) > len(suf)+1:
            return low[:-len(suf)]
    return low

# POS baseline: heuristicas por forma
def pos_baseline(token: str) -> str:
    t = token
    low = t.lower()
    if re.fullmatch(NUM_RE, t):
        return "NUM"
    if re.fullmatch(URL_RE, t) or re.fullmatch(HANDLE_RE, t) or re.fullmatch(HASHTAG_RE, t):
        return "X"
    if low in {"el","la","los","las","un","una","unos","unas","lo","al","del"}:
        return "DET"
    if low in {"y","o","u","pero","aunque","sino"}:
        return "CCONJ"
    if low in {"en","a","de","con","por","para","sin","sobre","entre","hasta"}:
        return "ADP"
    if low.endswith("mente"):
        return "ADV"
    if low.endswith(("ar","er","ir")) or low.endswith(("ando","iendo","ado","ido")):
        return "VERB"
    if t[0].isupper() and len(t) > 1:
        return "PROPN"
    if low.endswith(("o","a","os","as","cion","ciones")):
        return "NOUN"
    return "X"

def tag_sentence(tokens):
    return [(tok, pos_baseline(tok), lemmatize_simple(tok)) for tok in tokens]

for doc in CORPUS[:2]:
    toks = tokenize_es(doc["text"])
    print(doc["title"])
    print(tag_sentence(toks[:15]))
    print()


Economia: inflacion y empleo
[('La', 'DET', 'la'), ('economia', 'NOUN', 'economia'), ('espano', 'NOUN', 'espano'), ('la', 'DET', 'la'), ('registra', 'NOUN', 'registra'), ('un', 'DET', 'un'), ('descenso', 'NOUN', 'descenso'), ('de', 'ADP', 'de'), ('la', 'DET', 'la'), ('inflacion', 'NOUN', 'infla'), ('y', 'CCONJ', 'y'), ('una', 'DET', 'una'), ('ligera', 'NOUN', 'ligera'), ('mejora', 'NOUN', 'mejora'), ('del', 'DET', 'del')]

Deportes: liga de futbol
[('El', 'DET', 'el'), ('Real', 'PROPN', 'real'), ('Madrid', 'PROPN', 'madrid'), ('vencio', 'NOUN', 'vencio'), ('2', 'NUM', '2'), ('-', 'X', '-'), ('0', 'NUM', '0'), ('al', 'DET', 'al'), ('Valencia', 'PROPN', 'valencia'), ('en', 'ADP', 'en'), ('el', 'DET', 'el'), ('Santiago', 'PROPN', 'santiago'), ('Bernabeu', 'PROPN', 'bernabeu'), ('.', 'X', '.'), ('Vinicius', 'PROPN', 'viniciu')]




## 1.3) Representaciones: BoW y TF-IDF (desde cero)

Construimos el vocabulario, TF, DF, IDF y el vector TF-IDF. Luego probamos similaridad coseno.


In [46]:

def build_vocab(docs, lowercase=True, strip_acc=True):
    vocab = {}
    df = Counter()
    tokenized_docs = []
    for d in docs:
        toks = tokenize_es(d["text"], lowercase=lowercase, strip_acc=strip_acc)
        tokenized_docs.append(toks)
        for w in set(toks):
            df[w] += 1
    for i, w in enumerate(sorted(df)):
        vocab[w] = i
    return vocab, df, tokenized_docs

def compute_idf(df, N):
    # idf suavizado estilo BM25
    idf = {t: math.log((N - df[t] + 0.5)/(df[t] + 0.5) + 1) for t in df}
    return idf

def vectorize_tfidf(tokens, vocab, idf):
    tf = Counter(tokens)
    vec = [0.0] * len(vocab)
    for t, c in tf.items():
        if t in vocab and t in idf:
            vec[vocab[t]] = (c) * idf[t]
    return vec

def cosine(a, b):
    num = sum(x*y for x,y in zip(a,b))
    da = math.sqrt(sum(x*x for x in a))
    db = math.sqrt(sum(y*y for y in b))
    return 0.0 if da==0 or db==0 else num/(da*db)

VOCAB, DF, TOKENS = build_vocab(CORPUS)
IDF = compute_idf(DF, len(CORPUS))
VECTORS = [vectorize_tfidf(toks, VOCAB, IDF) for toks in TOKENS]
len(VOCAB), list(itertools.islice(VOCAB.items(), 10))


(109,
 [(',', 0),
  ('-', 1),
  ('.', 2),
  ('0', 3),
  ('2', 4),
  ('7', 5),
  ('9', 6),
  ('a', 7),
  ('actualizar', 8),
  ('advier', 9)])


## 1.4) IR: Mini-buscador (TF-IDF) y Ejercicio 1: BM25

Implementamos busqueda por coseno con TF-IDF y evaluamos con metricas clasicas.  
Ejercicio 1: completa la funcion `bm25_score` y la consulta `search_bm25` (tests al final).


## 1) IR — Mini-buscador con BM25

**Objetivo.** Implementar un *ranker* BM25 sobre un mini-corpus y comparar con TF-IDF.

**Qué aprenderás**
- Construcción de vocabulario y DF/IDF suavizado.
- Cálculo de longitud de documento y `avgdl`.
- Fórmula BM25 y ranking top-k.
- Métricas rápidas: `precision@k`, `nDCG@k`.

**Entradas / Salidas**
- *Entrada:* una consulta en texto libre.
- *Salida:* lista ordenada de tuplas `(score, doc_id, título)`.

**Pasos (resumen)**
1. Tokenizar y normalizar la consulta.
2. Calcular `BM25(q, d)` para cada documento (usa `IDF` y `AVGDL` ya construidos).
3. Ordenar por `score` descendente y devolver `topk`.

**Criterios de evaluación**
- El documento **“Clima: alerta por lluvias”** debe quedar top-1 para la consulta `lluvias intensas en el norte`.
- El código debe manejar consultas vacías o términos OOV sin fallar.

In [47]:
import math

# === Index bootstrap helpers (avoid NameError on VOCAB/IDF/VECTORS/TOKENS) ===
def init_index(lowercase=True, strip_acc=True):
    """Build corpus index: VOCAB, DF, TOKENS, IDF, VECTORS, AVGDL."""
    if "CORPUS" not in globals():
        raise RuntimeError("CORPUS is not defined. Define CORPUS before building the index.")
    # Requires: build_vocab, compute_idf, vectorize_tfidf
    global VOCAB, DF, TOKENS, IDF, VECTORS, AVGDL
    VOCAB, DF, TOKENS = build_vocab(CORPUS, lowercase=lowercase, strip_acc=strip_acc)
    IDF = compute_idf(DF, len(CORPUS))
    VECTORS = [vectorize_tfidf(toks, VOCAB, IDF) for toks in TOKENS]
    AVGDL = sum(len(t) for t in TOKENS) / len(TOKENS)

def ensure_index_ready():
    """Create the corpus index if any of the required globals is missing."""
    required = ("VOCAB", "DF", "TOKENS", "IDF", "VECTORS", "AVGDL")
    if any(name not in globals() for name in required):
        init_index()

# === TF-IDF search (now guarded with ensure_index_ready) ===
def search_tfidf(query, topk=5):
    ensure_index_ready()  # make sure VOCAB/IDF/VECTORS exist
    qtoks = tokenize_es(query, lowercase=True, strip_acc=True)
    qvec = vectorize_tfidf(qtoks, VOCAB, IDF)
    scores = []
    for d, vec in zip(CORPUS, VECTORS):
        scores.append((cosine(qvec, vec), d["id"], d["title"]))
    return sorted(scores, key=lambda x: x[0], reverse=True)[:topk]

def prec_at_k(results, relevant_ids, k=5):
    hits = sum(1 for score, docid, _ in results[:k] if docid in relevant_ids)
    return hits / k

def dcg_at_k(gains, k):
    return sum(g / math.log2(i+2) for i, g in enumerate(gains[:k]))

def ndcg_at_k(results, relevant_ids, k=5):
    gains = [1.0 if docid in relevant_ids else 0.0 for _, docid, _ in results]
    ideal = sorted(gains, reverse=True)
    return 0.0 if sum(ideal[:k]) == 0 else dcg_at_k(gains, k) / dcg_at_k(ideal, k)

# ===== EJERCICIO 1: Implementa BM25 =====
def bm25_score(query_tokens, doc_tokens, *, k1=1.5, b=0.75, idf, avgdl):
    """Return BM25 score of a query against a document.
    Keyword-only args 'idf' (dict) and 'avgdl' (float) are required.
    """
    tf = Counter(doc_tokens)
    dl = len(doc_tokens)
    score = 0.0
    for t in query_tokens:
        # skip OOV tokens or tokens not present in the doc
        if t not in idf:
            continue
        f = tf.get(t, 0)
        if f == 0:
            continue
        idf_t = idf[t]
        numer = f * (k1 + 1.0)
        denom = f + k1 * (1.0 - b + b * (dl / avgdl))
        score += idf_t * (numer / denom)
    return score

def search_bm25(query, topk=5, k1=1.5, b=0.75):
    """BM25 retrieval over the mini-corpus."""
    ensure_index_ready()  # make sure TOKENS/IDF/AVGDL exist
    qtoks = tokenize_es(query, lowercase=True, strip_acc=True)
    scores = []
    for d, doc_toks in zip(CORPUS, TOKENS):
        s = bm25_score(qtoks, doc_toks, k1=k1, b=b, idf=IDF, avgdl=AVGDL)
        scores.append((s, d["id"], d["title"]))
    return sorted(scores, key=lambda x: x[0], reverse=True)[:topk]

# --- Demo ---
q = "lluvias intensas en el norte"
print("Consulta:", q)
for s, i, t in search_tfidf(q, topk=3):
    print(f"{s:.3f} | {i} | {t}")

print("\nBM25 ->")
for s, i, t in search_bm25(q, topk=3):
    print(f"{s:.3f} | {i} | {t}")


Consulta: lluvias intensas en el norte
0.536 | 8 | Clima: alerta por lluvias
0.052 | 4 | Ciencia: energia solar
0.052 | 1 | Economia: inflacion y empleo

BM25 ->
7.612 | 8 | Clima: alerta por lluvias
1.473 | 1 | Economia: inflacion y empleo
1.465 | 4 | Ciencia: energia solar



## 1.5) IE: NER (baseline) y Ejercicio 2: F1 por solape parcial

Construimos un mini-dataset NER (BIO) y un baseline por patrones/gaceteros.  
Ejercicio 2: implementa `f1_span_overlap` para evaluar con solape parcial (util cuando la mencion detectada se solapa pero no coincide exactamente).


## 2) NER — Métrica con solape (IoU) y greedy matching

**Objetivo.** Evaluar entidades (BIO) permitiendo coincidencias **parciales** por IoU sobre spans.

**Qué aprenderás**
- Representar entidades como spans `(start, end, tipo)` en índices `[start, end)`.
- Calcular **IoU 1D** de spans y emparejar de forma **greedy** predicciones↔oro.
- Derivar `precision`, `recall` y `F1` con guardas ante división por cero.

**Entradas / Salidas**
- *Entrada:* `pred_spans` y `gold_spans` como listas de `tuplas`.
- *Salida:* `(precision, recall, f1)`.

**Pasos (resumen)**
1. Para cada predicción, buscar el oro no usado con **mayor IoU ≥ umbral** (y mismo tipo si `type_sensitive=True`).
2. Contabilizar **TP** por cada emparejamiento; los restantes son **FP/FN**.
3. Calcular métricas.

**Criterios de evaluación**
- En el *toy set* incluido, el F1 exacto debe ser 1.0 con el *baseline* tras impedir sobrescrituras.
- Con IoU < 1 se deben observar casos con `0 < F1 < 1`.

In [48]:
import re
from collections import Counter

# Mini dataset NER (BIO): (tokens, tags). Tipos: PER, ORG, LOC
NER_DATA = [
    (["El","Festival","de","San","Sebastian","inaugura","su","edicion","."],
     ["O","B-ORG","I-ORG","I-ORG","I-ORG","O","O","O","O"]),
    (["Vinicius","marco","el","primer","gol","del","partido","."],
     ["B-PER","O","O","O","O","O","O","O"]),
    (["La","Universidad","Rey","Juan","Carlos","financia","proyectos","."],
     ["O","B-ORG","I-ORG","I-ORG","I-ORG","O","O","O"]),
    (["Madrid","registra","lluvias","intensas","."],
     ["B-LOC","O","O","O","O"]),
]

GAZ_ORG = {"Festival de San Sebastian","Universidad Rey Juan Carlos"}
GAZ_LOC = {"Madrid","San Sebastian"}

def apply_span(tags, start, length, label):
    """Write BIO tags only if the target positions are currently 'O' (no overwrite)."""
    if any(tags[i] != "O" for i in range(start, start+length)):
        return  # do not overwrite existing labels
    tags[start] = f"B-{label}"
    for j in range(1, length):
        tags[start+j] = f"I-{label}"

def ner_baseline(tokens):
    """Gazetteer-first baseline with overwrite protection and simple capitalization heuristic for PER."""
    tags = ["O"] * len(tokens)
    text = " ".join(tokens)

    # ORG first (higher priority)
    for org in sorted(GAZ_ORG, key=len, reverse=True):
        m = re.search(r"\b"+re.escape(org)+r"\b", text)
        if m:
            start = text[:m.start()].count(" ")
            length = org.count(" ")+1
            apply_span(tags, start, length, "ORG")

    # LOC second (only if slots are still 'O')
    for loc in sorted(GAZ_LOC, key=len, reverse=True):
        m = re.search(r"\b"+re.escape(loc)+r"\b", text)
        if m:
            start = text[:m.start()].count(" ")
            length = loc.count(" ")+1
            apply_span(tags, start, length, "LOC")

    # Single capitalized tokens as PER (only where still 'O')
    for i, t in enumerate(tokens):
        if tags[i] == "O" and t and t[0].isupper() and t.lower() not in {"el","la","los","las"}:
            tags[i] = "B-PER"
    return tags

def bio_spans(tags):
    """Convert BIO tag sequence to list of spans (start, end, type), half-open [start, end)."""
    spans = []
    i = 0
    while i < len(tags):
        if tags[i].startswith("B-"):
            typ = tags[i][2:]
            j = i+1
            while j < len(tags) and tags[j] == "I-"+typ:
                j += 1
            spans.append((i, j, typ))  # [i, j)
            i = j
        else:
            i += 1
    return spans

def f1_exact(pred_spans, gold_spans):
    """Exact match F1 over spans including type."""
    ps = set(pred_spans); gs = set(gold_spans)
    tp = len(ps & gs); fp = len(ps - gs); fn = len(gs - ps)
    prec = tp/(tp+fp) if tp+fp>0 else 0.0
    rec = tp/(tp+fn) if tp+fn>0 else 0.0
    f1 = 2*prec*rec/(prec+rec) if prec+rec>0 else 0.0
    return prec, rec, f1

# ===== EXERCISE 2: F1 with partial overlap =====
def iou(span1, span2):
    """1D IoU of half-open spans [i, j). Types are ignored here."""
    (a1,a2),(b1,b2) = (span1[:2], span2[:2])
    inter = max(0, min(a2,b2)-max(a1,b1))
    union = max(a2,b2)-min(a1,b1)
    return inter/union if union>0 else 0.0

def f1_span_overlap(pred_spans, gold_spans, iou_threshold=0.5, type_sensitive=True):
    """Compute precision/recall/F1 allowing partial matches via IoU.
    Greedy matching: for each prediction, pair with the best unmatched gold (max IoU ≥ threshold).
    Span format: (start, end, type), indices are half-open [start, end).
    """
    used_gold = set()
    tp = 0

    for ps in pred_spans:
        best_iou = -1.0
        best_j = None
        for j, gs in enumerate(gold_spans):
            if j in used_gold:
                continue
            if type_sensitive and ps[2] != gs[2]:
                continue
            iou_val = iou(ps, gs)
            if iou_val >= iou_threshold and iou_val > best_iou:
                best_iou = iou_val
                best_j = j
        if best_j is not None:
            used_gold.add(best_j)
            tp += 1

    fp = len(pred_spans) - tp
    fn = len(gold_spans) - tp
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return prec, rec, f1

# --- Quick demo ---
# Exact-match scores per sentence
exact_scores = []
for tokens, gold_tags in NER_DATA:
    pred_tags = ner_baseline(tokens)
    exact_scores.append(
        f1_exact(bio_spans(pred_tags), bio_spans(gold_tags))
    )
print("Exact-match per sentence:", exact_scores)

# Overlap F1 (IoU>=0.5) per sentence
overlap_scores = []
for tokens, gold_tags in NER_DATA:
    pred_tags = ner_baseline(tokens)
    overlap_scores.append(
        f1_span_overlap(bio_spans(pred_tags), bio_spans(gold_tags), iou_threshold=0.5, type_sensitive=True)
    )
print("Overlap (IoU>=0.5) per sentence:", overlap_scores)


Exact-match per sentence: [(1.0, 1.0, 1.0), (1.0, 1.0, 1.0), (1.0, 1.0, 1.0), (1.0, 1.0, 1.0)]
Overlap (IoU>=0.5) per sentence: [(1.0, 1.0, 1.0), (1.0, 1.0, 1.0), (1.0, 1.0, 1.0), (1.0, 1.0, 1.0)]


In [49]:
# --- Example to show difference between exact-match and IoU-based scoring ---

# Tokens for display only (spans use indices on this list)
tokens = ["El","Festival","de","San","Sebastian","inaugura","."]

# Gold: ORG = "Festival de San Sebastian" -> [1:5)
gold_spans = [(1, 5, "ORG")]

# Prediction (partial): misses the last token "Sebastian" -> [1:4)
pred_spans_partial = [(1, 4, "ORG")]

def span_str(tokens, span):
    i, j, typ = span
    return f"{typ}: «{' '.join(tokens[i:j])}» [{i}:{j}]"

print("Tokens: ", " ".join(tokens))
print("Gold:   ", span_str(tokens, gold_spans[0]))
print("Pred:   ", span_str(tokens, pred_spans_partial[0]))

# 1) Exact-match (should fail -> F1=0.0)
p_exact, r_exact, f1_exact_val = f1_exact(pred_spans_partial, gold_spans)
print("\nExact-match:")
print(f"  Precision={p_exact:.3f}  Recall={r_exact:.3f}  F1={f1_exact_val:.3f}")

# 2) Overlap with IoU >= 0.5 (should succeed -> F1=1.0 because IoU=0.75)
p_iou05, r_iou05, f1_iou05 = f1_span_overlap(pred_spans_partial, gold_spans, iou_threshold=0.5, type_sensitive=True)
print("\nIoU-based (threshold=0.5, type_sensitive=True):")
print(f"  Precision={p_iou05:.3f}  Recall={r_iou05:.3f}  F1={f1_iou05:.3f}")

# 3) Stricter overlap with IoU >= 0.8 (should fail -> below threshold)
p_iou08, r_iou08, f1_iou08 = f1_span_overlap(pred_spans_partial, gold_spans, iou_threshold=0.8, type_sensitive=True)
print("\nIoU-based (threshold=0.8, type_sensitive=True):")
print(f"  Precision={p_iou08:.3f}  Recall={r_iou08:.3f}  F1={f1_iou08:.3f}")

# 4) Type mismatch case: same span as 2) but wrong type (ORG vs PER)
pred_spans_type_mismatch = [(1, 4, "PER")]
p_typ, r_typ, f1_typ = f1_span_overlap(pred_spans_type_mismatch, gold_spans, iou_threshold=0.5, type_sensitive=True)
print("\nIoU-based with type mismatch (threshold=0.5, type_sensitive=True):")
print(f"  Precision={p_typ:.3f}  Recall={r_typ:.3f}  F1={f1_typ:.3f}")

# If you relax type sensitivity, it will count as a match:
p_typ_rel, r_typ_rel, f1_typ_rel = f1_span_overlap(pred_spans_type_mismatch, gold_spans, iou_threshold=0.5, type_sensitive=False)
print("IoU-based with type mismatch (threshold=0.5, type_sensitive=False):")
print(f"  Precision={p_typ_rel:.3f}  Recall={r_typ_rel:.3f}  F1={f1_typ_rel:.3f}")


Tokens:  El Festival de San Sebastian inaugura .
Gold:    ORG: «Festival de San Sebastian» [1:5]
Pred:    ORG: «Festival de San» [1:4]

Exact-match:
  Precision=0.000  Recall=0.000  F1=0.000

IoU-based (threshold=0.5, type_sensitive=True):
  Precision=1.000  Recall=1.000  F1=1.000

IoU-based (threshold=0.8, type_sensitive=True):
  Precision=0.000  Recall=0.000  F1=0.000

IoU-based with type mismatch (threshold=0.5, type_sensitive=True):
  Precision=0.000  Recall=0.000  F1=0.000
IoU-based with type mismatch (threshold=0.5, type_sensitive=False):
  Precision=1.000  Recall=1.000  F1=1.000



## 1.2 Sintaxis de dependencias: toy parser shift-reduce y UAS/LAS

Mini conjunto anotado y un parser determinista (reglas por POS) para ilustrar el calculo de UAS/LAS.


In [50]:

@dataclass
class DepSentence:
    tokens: list
    pos: list
    heads: list  # indice del head (0-based); -1 para ROOT
    rels: list   # etiquetas

# Tres oraciones simples
SENTS = [
    DepSentence(
        tokens=["Juan","come","manzanas","."],
        pos   =["PROPN","VERB","NOUN","PUNCT"],
        heads =[1, -1, 1, 1],      # head de 'Juan' es 'come'(1), 'come' es ROOT(-1)
        rels  =["nsubj","root","obj","punct"],
    ),
    DepSentence(
        tokens=["La","universidad","financia","proyectos","."],
        pos   =["DET","NOUN","VERB","NOUN","PUNCT"],
        heads =[1, 2, -1, 2, 2],
        rels  =["det","nsubj","root","obj","punct"],
    ),
    DepSentence(
        tokens=["Maria","viajo","a","Madrid","en","2019","."],
        pos   =["PROPN","VERB","ADP","PROPN","ADP","NUM","PUNCT"],
        heads =[1, -1, 3, 1, 5, 1, 1],
        rels  =["nsubj","root","case","obl","case","obl","punct"],
    ),
]

def toy_parser(tokens, pos):
    """
    Parser determinista: asume VERB como head central, NOUN/PROPN a la izquierda -> nsubj, a la derecha -> obj,
    ADP+PROPN/NUM -> obl.
    Devuelve heads, rels.
    """
    n = len(tokens)
    # raiz = primer VERB o 0 si no hay
    try:
        root = pos.index("VERB")
    except ValueError:
        root = 0
    heads = [root]*n
    rels  = ["dep"]*n
    heads[root] = -1; rels[root] = "root"
    for i,(t,p) in enumerate(zip(tokens,pos)):
        if i==root: 
            continue
        if p in {"PROPN","NOUN"} and i < root:
            heads[i] = root; rels[i] = "nsubj"
        elif p in {"PROPN","NOUN"} and i > root:
            heads[i] = root; rels[i] = "obj"
        elif p=="ADP":
            # asocia con el siguiente NOUN/PROPN/NUM si existe; ese ira al verbo como obl
            j = i+1
            if j<n and pos[j] in {"PROPN","NOUN","NUM"}:
                heads[i] = j; rels[i] = "case"
                heads[j] = root; rels[j] = "obl"
        elif p=="PUNCT":
            heads[i] = root; rels[i] = "punct"
        else:
            heads[i] = root; rels[i] = "dep"
    return heads, rels

def uas_las(pred_h, pred_r, gold_h, gold_r):
    uas = sum(ph==gh for ph,gh in zip(pred_h, gold_h)) / len(gold_h)
    las = sum((ph==gh) and (pr==gr) for ph,pr,gh,gr in zip(pred_h, pred_r, gold_h, gold_r)) / len(gold_h)
    return uas, las

scores = []
for s in SENTS:
    ph, pr = toy_parser(s.tokens, s.pos)
    scores.append(uas_las(ph, pr, s.heads, s.rels))
scores


[(1.0, 1.0), (0.8, 0.8), (1.0, 0.7142857142857143)]

### Soluciones (se cargan si `USE_SOLUTIONS=True`)

In [51]:

def _bm25_score_solution(query_tokens, doc_tokens, k1=1.5, b=0.75, idf=None, avgdl=None):
    if avgdl is None:
        raise ValueError("avgdl debe ser pasado")
    if idf is None:
        raise ValueError("idf requerido")
    tf = Counter(doc_tokens)
    dl = len(doc_tokens)
    score = 0.0
    for t in query_tokens:
        if t not in idf or t not in tf:
            continue
        idf_t = idf[t]
        numer = tf[t]*(k1+1)
        denom = tf[t] + k1*(1 - b + b*dl/avgdl)
        score += idf_t * (numer/denom)
    return score

def _search_bm25_solution(query, topk=5, k1=1.5, b=0.75):
    qtoks = tokenize_es(query, lowercase=True, strip_acc=True)
    avgdl = sum(len(t) for t in TOKENS) / len(TOKENS)
    scores = []
    for d, doc_toks in zip(CORPUS, TOKENS):
        s = _bm25_score_solution(qtoks, doc_toks, k1=k1, b=b, idf=IDF, avgdl=avgdl)
        scores.append((s, d["id"], d["title"]))
    return sorted(scores, reverse=True)[:topk]

def _f1_span_overlap_solution(pred_spans, gold_spans, iou_threshold=0.5, type_sensitive=True):
    used_gold = set()
    tp = 0
    for ps in pred_spans:
        best = -1.0; best_j = None
        for j, gs in enumerate(gold_spans):
            if j in used_gold:
                continue
            if type_sensitive and ps[2] != gs[2]:
                continue
            iou_val = iou(ps, gs)
            if iou_val >= iou_threshold and iou_val > best:
                best = iou_val; best_j = j
        if best_j is not None:
            used_gold.add(best_j)
            tp += 1
    fp = len(pred_spans) - tp
    fn = len(gold_spans) - tp
    prec = tp/(tp+fp) if tp+fp>0 else 0.0
    rec = tp/(tp+fn) if tp+fn>0 else 0.0
    f1 = 2*prec*rec/(prec+rec) if prec+rec>0 else 0.0
    return prec, rec, f1

if USE_SOLUTIONS:
    bm25_score = _bm25_score_solution
    search_bm25 = _search_bm25_solution
    f1_span_overlap = _f1_span_overlap_solution

# Tests automaticos basicos
def _run_tests():
    # BM25 ranking
    q = "lluvias intensas en el norte"
    r_bm25 = search_bm25(q, topk=3)
    assert r_bm25[0][1] == 8, "BM25 deberia rankear primero el doc 8 (lluvias)"
    # F1 overlap: span parcial
    pred = [(0,2,"ORG")]     # 'El Festival' (parcial del gold 'Festival de San Sebastian')
    gold = [(1,5,"ORG")]
    p,r,f = f1_span_overlap(pred, gold, iou_threshold=0.3, type_sensitive=True)
    assert f > 0 and f < 1, "El F1 por solape deberia ser intermedio (0<f<1)."
    print("Tests OK.")

if USE_SOLUTIONS:
    _run_tests()



## 2) Demostraciones rapidas

Ejecuta esta celda para ver el buscador TF-IDF vs BM25 y la evaluacion de NER.


In [52]:

# Demo IR
q = "presupuesto del congreso y oposicion al gasto"
print("TF-IDF ->")
for s, i, t in search_tfidf(q, topk=5):
    print(f"{s:.3f} | {i} | {t}")
print("\nBM25 ->")
try:
    for s, i, t in search_bm25(q, topk=5):
        print(f"{s:.3f} | {i} | {t}")
except Exception as e:
    print("BM25 aun no implementado:", e)

# Demo NER
for tokens, gold_tags in NER_DATA:
    pred_tags = ner_baseline(tokens)
    print("\n", " ".join(tokens))
    print("G:", gold_tags)
    print("P:", pred_tags)
    p1,r1,f1 = f1_exact(bio_spans(pred_tags), bio_spans(gold_tags))
    print(f"F1 exacto: {f1:.3f}")


TF-IDF ->
0.475 | 6 | Politica: presupuesto
0.123 | 2 | Deportes: liga de futbol
0.069 | 1 | Economia: inflacion y empleo
0.059 | 5 | Salud: habitos y bienestar
0.019 | 8 | Clima: alerta por lluvias

BM25 ->
7.674 | 6 | Politica: presupuesto
2.426 | 2 | Deportes: liga de futbol
1.650 | 1 | Economia: inflacion y empleo
1.366 | 5 | Salud: habitos y bienestar
0.775 | 8 | Clima: alerta por lluvias

 El Festival de San Sebastian inaugura su edicion .
G: ['O', 'B-ORG', 'I-ORG', 'I-ORG', 'I-ORG', 'O', 'O', 'O', 'O']
P: ['O', 'B-ORG', 'I-ORG', 'I-ORG', 'I-ORG', 'O', 'O', 'O', 'O']
F1 exacto: 1.000

 Vinicius marco el primer gol del partido .
G: ['B-PER', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
P: ['B-PER', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
F1 exacto: 1.000

 La Universidad Rey Juan Carlos financia proyectos .
G: ['O', 'B-ORG', 'I-ORG', 'I-ORG', 'I-ORG', 'O', 'O', 'O']
P: ['O', 'B-ORG', 'I-ORG', 'I-ORG', 'I-ORG', 'O', 'O', 'O']
F1 exacto: 1.000

 Madrid registra lluvias intensas .
G: ['B-LOC', 'O',

## 3) WSD — Lesk simplificado

**Objetivo.** Desambiguar palabras ambiguas comparando el **solape** entre el **contexto** y **glosas**.

**Qué aprenderás**
- Normalización (minúsculas + eliminación de tildes) y filtrado de *stopwords*.
- Construcción de conjuntos léxicos y cómputo de intersecciones.
- *Baselines* (primer sentido) y *sanity checks* de `accuracy`.

**Entradas / Salidas**
- *Entrada:* (palabra objetivo, oración) + diccionario de glosas.
- *Salida:* clave de sentido seleccionada.

In [53]:

# Helpers and small resources for WSD
import re, unicodedata
from collections import Counter

STOP = {"el","la","los","las","un","una","unos","unas","de","del","al","y","o","u","en","a","con","por","para","sin","sobre"}

def strip_accents(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

def tokenize(text):
    return re.findall(r"[\wáéíóúñÁÉÍÓÚÑ]+", text, flags=re.UNICODE)

def norm(tokens):
    return [strip_accents(t.lower()) for t in tokens]

# Glosses and dataset
GLOSSES = {
    "banco": {
        "asiento": "Asiento largo de madera o piedra donde se sientan varias personas.",
        "financiero": "Entidad de credito o institucion financiera dedicada a depositos, prestamos y servicios bancarios."
    },
    "raton": {
        "animal": "Pequeno mamifero roedor con bigotes y cola larga.",
        "dispositivo": "Dispositivo periferico de computadora que controla el puntero en la pantalla."
    }
}

WSD_DATA = [
    ("banco", "Se sento en el banco del parque para leer.", "asiento"),
    ("banco", "El banco aprobo el prestamo hipotecario.", "financiero"),
    ("banco", "Nos reunimos en un banco de piedra frente al lago.", "asiento"),
    ("banco", "El banco central anuncio nuevas medidas.", "financiero"),
    ("raton", "El raton mordisqueo el queso en la cocina.", "animal"),
    ("raton", "Mi raton inalambrico necesita pilas nuevas.", "dispositivo"),
    ("raton", "El gato persiguio al raton por el jardin.", "animal"),
    ("raton", "No me funciona el raton del ordenador.", "dispositivo"),
]

def lesk_disambiguate(context_tokens, target, glosses):
    """Simplified Lesk: choose the sense with the largest overlap of content words between
    the (normalized, stopword-filtered) sentence and the gloss text.
    """
    ctx = set(w for w in norm(context_tokens) if w not in STOP)
    best = None; best_s = -1
    for sense, gloss in glosses[target].items():
        gset = set(w for w in norm(tokenize(gloss)) if w not in STOP)
        s = len(ctx & gset)
        if s > best_s:
            best = sense; best_s = s
    return best

def accuracy_wsd(dataset, glosses):
    correct = 0
    for target, sent, gold in dataset:
        pred = lesk_disambiguate(tokenize(sent), target, glosses)
        correct += int(pred == gold)
    return correct / len(dataset)

def baseline_first_sense(dataset, glosses):
    first = {t: next(iter(glosses[t].keys())) for t,_,_ in dataset}
    return sum(int(first[t] == gold) for t,_,gold in dataset) / len(dataset)

print("WSD baseline (first sense):", baseline_first_sense(WSD_DATA, GLOSSES))
print("WSD Lesk accuracy:", accuracy_wsd(WSD_DATA, GLOSSES))


WSD baseline (first sense): 0.5
WSD Lesk accuracy: 0.5


### Mejora de Lesk y comparación

- **Lesk (BoW + alias + lematización ligera)**: usa bolsa de palabras, mapea *ordenador→computadora* y recorta plurales simples.
- **Lesk (ventana)**: restringe el contexto a una **ventana** alrededor del término objetivo.

In [54]:

from collections import Counter

def lesk_disambiguate_bow(context_tokens, target, glosses):
    """Lesk with bag-of-words overlap + tiny lemmatization and alias mapping."""
    def normalize(tokens):
        return [strip_accents(t.lower()) for t in tokens]
    def simple_lemma(w):
        return w[:-1] if len(w) > 4 and w.endswith('s') else w
    ALIAS = {"ordenador":"computadora"}
    def canon(w):
        w = ALIAS.get(w, w)
        return simple_lemma(w)

    tgt = canon(strip_accents(target.lower()))
    ctx = [canon(w) for w in normalize(context_tokens) if w not in STOP]
    ctx = [w for w in ctx if w != tgt]
    ctx_bow = Counter(ctx)

    best, best_s = None, -1
    for sense, gloss in glosses[target].items():
        g = [canon(w) for w in normalize(tokenize(gloss)) if w not in STOP]
        g_bow = Counter(g)
        s = sum(min(ctx_bow[w], g_bow[w]) for w in g_bow)
        if s > best_s:
            best, best_s = sense, s
    return best

def lesk_disambiguate_window(sent_tokens, target, glosses, window=5):
    """Lesk variant that narrows the context window around the first target occurrence."""
    tokens = tokenize(" ".join(sent_tokens))
    norm_tokens = [strip_accents(t.lower()) for t in tokens]
    tgt = strip_accents(target.lower())
    try:
        i = norm_tokens.index(tgt)
        left = max(0, i - window)
        right = min(len(tokens), i + window + 1)
        context_tokens = tokens[left:right]
    except ValueError:
        context_tokens = tokens
    return lesk_disambiguate_bow(context_tokens, target, glosses)

# Accuracy comparison
baseline = baseline_first_sense(WSD_DATA, GLOSSES)
acc_set = accuracy_wsd(WSD_DATA, GLOSSES)
acc_bow = sum(lesk_disambiguate_bow(tokenize(sent), target, GLOSSES) == gold
              for target, sent, gold in WSD_DATA) / len(WSD_DATA)
acc_win = sum(lesk_disambiguate_window(tokenize(sent), target, GLOSSES, window=5) == gold
              for target, sent, gold in WSD_DATA) / len(WSD_DATA)

print(f"Baseline (first sense): {baseline:.3f}")
print(f"Lesk (set overlap):    {acc_set:.3f}")
print(f"Lesk (BoW+alias+lem):  {acc_bow:.3f}")
print(f"Lesk (windowed BoW):   {acc_win:.3f}")

# Side-by-side example
target = "raton"
sent = "No me funciona el raton del ordenador."
ctx = tokenize(sent)
pred_set = lesk_disambiguate(ctx, target, GLOSSES)
pred_bow = lesk_disambiguate_bow(ctx, target, GLOSSES)
pred_win = lesk_disambiguate_window(ctx, target, GLOSSES, window=5)
print("\nEjemplo comparativo:")
print("  Oración:", sent)
print("  Objetivo:", target)
print("  Oro esperado: dispositivo")
print("  Lesk (set):   ", pred_set)
print("  Lesk (BoW):   ", pred_bow)
print("  Lesk (win):   ", pred_win)


Baseline (first sense): 0.500
Lesk (set overlap):    0.500
Lesk (BoW+alias+lem):  0.750
Lesk (windowed BoW):   0.750

Ejemplo comparativo:
  Oración: No me funciona el raton del ordenador.
  Objetivo: raton
  Oro esperado: dispositivo
  Lesk (set):    animal
  Lesk (BoW):    dispositivo
  Lesk (win):    dispositivo


## 4) Correferencia de pronombres — heurística de recencia

**Objetivo.** Resolver pronombres por **recencia + concordancia** (género desconocido = comodín) y evaluar con **F1 por pares**.

In [57]:

# Coreference data and solver (heuristic)
TEXTS = [
    (["Maria","vio","a","Ana","y","luego","ella","llamo","a","su","madre","."],
     [(0,1,"NP"), (3,4,"NP"), (6,7,"PRON"), (9,10,"PRON"), (9,11,"NP-POSS")],
     [{0,2}, {3,4}]  # {Maria, ella}, {Ana, su madre}
    ),
    (["Carlos","dijo","a","Luis","que","el","llegaria","tarde","."],
     [(0,1,"NP"), (3,4,"NP"), (5,6,"PRON")],
     [{0,2}]  # 'el' -> Carlos
    ),
]

GENDER_LEX = {"maria":"F", "ana":"F", "madre":"F", "carlos":"M", "luis":"M"}
PRON_INFO = {"ella":"F", "el":"M", "lo":"M", "la":"F", "le":"X", "su":"X"}  # X = unknown/neutral

def pairwise_pairs_from_clusters(clusters):
    pairs = set()
    for cl in clusters:
        cl = sorted(list(cl))
        for i in range(len(cl)):
            for j in range(i+1, len(cl)):
                pairs.add((cl[i], cl[j]))
    return pairs

def pairwise_f1(pred_clusters, gold_clusters):
    p_pairs = pairwise_pairs_from_clusters(pred_clusters)
    g_pairs = pairwise_pairs_from_clusters(gold_clusters)
    tp = len(p_pairs & g_pairs)
    fp = len(p_pairs - g_pairs)
    fn = len(g_pairs - p_pairs)
    prec = tp/(tp+fp) if tp+fp>0 else 0.0
    rec  = tp/(tp+fn) if tp+fn>0 else 0.0
    f1   = 2*prec*rec/(prec+rec) if prec+rec>0 else 0.0
    return prec, rec, f1

def resolve_coref_pronouns(tokens, mentions):
    """Nearest-previous NP matching gender (if known). Merge clusters greedily."""
    clusters = [set([i]) for i in range(len(mentions))]

    def gender_of_text(txt):
        w = txt.split()[-1].lower()
        return GENDER_LEX.get(w, "X")

    for i, (s,e,label) in enumerate(mentions):
        if "PRON" in label:
            txt = " ".join(tokens[s:e]).lower()
            pgen = PRON_INFO.get(txt, "X")
            best_j = None
            for j in range(i-1, -1, -1):
                if "NP" in mentions[j][2] and "PRON" not in mentions[j][2]:
                    cand_txt = " ".join(tokens[mentions[j][0]:mentions[j][1]])
                    g = gender_of_text(cand_txt)
                    if pgen == "X" or g == "X" or g == pgen:
                        best_j = j
                        break
            if best_j is not None:
                # merge clusters containing i and best_j
                ci = next(k for k,cl in enumerate(clusters) if i in cl)
                cj = next(k for k,cl in enumerate(clusters) if best_j in cl)
                if ci != cj:
                    clusters[cj] |= clusters[ci]
                    clusters.pop(ci)
    return clusters

def evaluate_coref(texts):
    return [pairwise_f1(resolve_coref_pronouns(tokens, mentions), gold)
            for tokens, mentions, gold in texts]

print("Coref pairwise F1 scores:", evaluate_coref(TEXTS))


Coref pairwise F1 scores: [(0.0, 0.0, 0.0), (0.0, 0.0, 0.0)]


In [58]:
# --- Improved coreference resolver + comparison (keeps your baseline intact) ---

def resolve_coref_pronouns_improved(tokens, mentions):
    """Greedy pronoun resolver with two simple heuristics:
       (H1) Skip NPs immediately preceded by 'a' (likely direct object in Spanish).
       (H2) If a 'su' PRON is followed by an NP-POSS starting at the same index, merge them.
    """
    clusters = [set([i]) for i in range(len(mentions))]

    def gender_of_text(txt):
        # approximate gender by last token lookup
        w = txt.split()[-1].lower()
        return GENDER_LEX.get(w, "X")

    def merge(i, j):
        # merge clusters containing indices i and j
        ci = next(k for k, cl in enumerate(clusters) if i in cl)
        cj = next(k for k, cl in enumerate(clusters) if j in cl)
        if ci != cj:
            clusters[cj] |= clusters[ci]
            clusters.pop(ci)

    # H2: glue 'su' with a following NP-POSS that starts at the same span start
    for i, (s, e, label) in enumerate(mentions):
        if "PRON" in label:
            txt = " ".join(tokens[s:e]).lower()
            if txt == "su":
                for j in range(i+1, len(mentions)):
                    s2, e2, lab2 = mentions[j]
                    if s2 == s and "NP-POSS" in lab2:
                        merge(i, j)
                        break

    # Main pronoun resolution with H1 skip rule and gender agreement
    for i, (s, e, label) in enumerate(mentions):
        if "PRON" in label:
            txt = " ".join(tokens[s:e]).lower()
            pgen = PRON_INFO.get(txt, "X")
            best_j = None
            for j in range(i-1, -1, -1):
                s_j, e_j, lab_j = mentions[j]
                if "NP" in lab_j and "PRON" not in lab_j:
                    # H1: skip NP if immediately preceded by 'a' (object marker)
                    if s_j - 1 >= 0 and tokens[s_j - 1].lower() == "a":
                        continue
                    cand_txt = " ".join(tokens[s_j:e_j])
                    g = gender_of_text(cand_txt)
                    # unknown 'X' as wildcard
                    if pgen == "X" or g == "X" or g == pgen:
                        best_j = j
                        break
            if best_j is not None:
                merge(i, best_j)

    return clusters

def evaluate_coref_with(resolver, texts):
    """Evaluate a resolver on TEXTS using pairwise F1 (requires pairwise_f1)."""
    return [pairwise_f1(resolver(tokens, mentions), gold)
            for tokens, mentions, gold in texts]

# --- Comparison: baseline vs improved ---
print("Baseline resolver (pairwise F1):", evaluate_coref_with(resolve_coref_pronouns, TEXTS))
print("Improved resolver (pairwise F1):", evaluate_coref_with(resolve_coref_pronouns_improved, TEXTS))


Baseline resolver (pairwise F1): [(0.0, 0.0, 0.0), (0.0, 0.0, 0.0)]
Improved resolver (pairwise F1): [(0.3333333333333333, 1.0, 0.5), (1.0, 1.0, 1.0)]


## 5) QA factoidal mini — extracción SVO

**Objetivo.** Responder “¿Quién VERBO OBJETO?” extrayendo un **Sujeto-Verbo-Objeto** mínimo.

In [59]:

# QA mini via simple SVO extraction
DATA_QA = [
    (["Juan","compro","un","libro","."], "compro un libro", "Juan"),
    (["Maria","visito","Madrid","."], "visito Madrid", "Maria"),
    (["El","banquero","apoyo","el","presupuesto","."], "apoyo el presupuesto", "El banquero"),
]

def extract_svo(tokens):
    """Very simple heuristic extractor."""
    if not tokens:
        return ("","","")
    if tokens[0].lower() in {"el","la","los","las"} and len(tokens) > 1:
        subj = tokens[0] + " " + tokens[1]
        verb = tokens[2] if len(tokens) > 2 else ""
        obj = " ".join(tokens[3:]).strip(" .!?,;:")
    else:
        subj = tokens[0]
        verb = tokens[1] if len(tokens) > 1 else ""
        obj = " ".join(tokens[2:]).strip(" .!?,;:")
    return (subj, verb, obj)

def answer_who_did(tokens, query):
    subj, v, o = extract_svo(tokens)
    q = " ".join(query.lower().split())
    vo = (v + " " + o).lower().strip()
    return subj if vo == q else ""

def evaluate_qa(dataset):
    ok = 0
    for tokens, q, gold in dataset:
        pred = answer_who_did(tokens, q)
        ok += int(pred == gold)
    return ok/len(dataset)

print("QA accuracy:", evaluate_qa(DATA_QA))


QA accuracy: 1.0


In [60]:
# --- Counterexamples for current SVO QA heuristic (expected to fail) ---

cases = [
    # 1) Clitic pronoun -> missing explicit object ("lo" instead of "un libro")
    (["Juan","lo","compro","."], "compro un libro", "Juan", "clitic pronoun 'lo' (no explicit object)"),
    # 2) Passive voice -> subject/object inverted wrt active query
    (["El","libro","fue","comprado","por","Juan","."], "compro un libro", "Juan", "passive voice"),
    # 3) Object with modifier -> exact string mismatch
    (["Juan","compro","un","libro","nuevo","."], "compro un libro", "Juan", "object has extra modifier ('nuevo')")
]

print("Counterexamples (current answer_who_did should fail):\n")
for tokens, query, gold, reason in cases:
    pred = answer_who_did(tokens, query)
    print("Tokens: ", " ".join(tokens))
    print("Query:  ", query)
    print("Gold:   ", gold)
    print("Reason: ", reason)
    print("Pred:   ", repr(pred))   
    print("-"*60)


Counterexamples (current answer_who_did should fail):

Tokens:  Juan lo compro .
Query:   compro un libro
Gold:    Juan
Reason:  clitic pronoun 'lo' (no explicit object)
Pred:    ''
------------------------------------------------------------
Tokens:  El libro fue comprado por Juan .
Query:   compro un libro
Gold:    Juan
Reason:  passive voice
Pred:    ''
------------------------------------------------------------
Tokens:  Juan compro un libro nuevo .
Query:   compro un libro
Gold:    Juan
Reason:  object has extra modifier ('nuevo')
Pred:    ''
------------------------------------------------------------
